In [2]:
import os, glob, gc, re
import numpy as np
import xarray as xr
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# ============================================================
# 导入计算函数
# ============================================================
try:
    from aostools_functions import ComputeEPfluxDiv
except ImportError:
    raise ImportError("请确保 'aostools_functions.py' 与此脚本在同一目录下！")

# ============================================================
# 批量指定多个已经处理好的 *_timefixed 目录
# ============================================================
ROOT_DIRS = [
    "/mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed",
    "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002_timefixed",
    "/mnt/soclim0/public_data/weiji/B2000WCN007009010011_Clim3D_timefixed",
    "/mnt/soclim0/public_data/weiji/BWCN",
]

# ============================================================
# Settings
# ============================================================
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

DO_UBAR     = False
WAVE_ALL    = -1
WAVE_1      = 1
WAVE_2      = 2
MAX_WORKERS = 32
OVERWRITE   = False

YEAR_RE = re.compile(r"\.cam\.h3\.(\d{4})\.")

# ============================================================
# Helpers
# ============================================================
def parse_year_from_filename(path: str) -> int:
    m = YEAR_RE.search(os.path.basename(path))
    if not m:
        raise ValueError(f"Cannot parse year from filename: {path}")
    return int(m.group(1))

def detect_prefix(var_dir: str, var_name: str) -> str:
    files = sorted(glob.glob(os.path.join(var_dir, f"*.{var_name}.nc")))
    if not files:
        raise FileNotFoundError(f"No files found in {var_dir} for {var_name}")
    base = os.path.basename(files[0])
    m = re.match(rf"(.+)\.(\d{{4}})\.{re.escape(var_name)}\.nc$", base)
    if not m:
        raise ValueError(f"Cannot detect prefix from filename: {base}")
    return m.group(1)

def file_for(var_dir: str, year_int: int, var_name: str) -> str:
    y = f"{year_int:04d}"
    candidates = glob.glob(os.path.join(var_dir, f"*.cam.h3.{y}.{var_name}.nc"))
    if len(candidates) == 0:
        raise FileNotFoundError(f"Missing {var_name} file for year {y} in {var_dir}")
    if len(candidates) > 1:
        raise RuntimeError(f"Multiple candidates found for {var_name} year {y}: {candidates}")
    return candidates[0]

def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    return ds["hyam"] * ds["P0"] + ds["hybm"] * ds["PS"]

def interp_profile_logp_4d(v_hyb, p_hyb, p_tgt_pa):
    p_tgt_pa = np.asarray(p_tgt_pa, float)

    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full(p_tgt_pa.shape, np.nan, float)
        p_use, v_use = pcol[m], vcol[m]
        idx = np.argsort(p_use)
        return np.interp(
            np.log(p_tgt_pa),
            np.log(p_use[idx]),
            v_use[idx],
            left=np.nan,
            right=np.nan
        )

    dask_gufunc_kwargs = {"output_sizes": {"plev": len(p_tgt_pa)}}
    out = xr.apply_ufunc(
        _interp_col, v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True, dask="parallelized", output_dtypes=[float],
        dask_gufunc_kwargs=dask_gufunc_kwargs
    )
    return out.assign_coords(plev=("plev", p_tgt_pa))

def get_available_years(U_DIR, V_DIR, T_DIR, OMEGA_DIR):
    """
    自动取 U/V/T/OMEGA 四个变量共同存在的年份交集
    """
    year_sets = []
    for var_dir, var_name in [
        (U_DIR, "U"),
        (V_DIR, "V"),
        (T_DIR, "T"),
        (OMEGA_DIR, "OMEGA"),
    ]:
        files = sorted(glob.glob(os.path.join(var_dir, f"*.{var_name}.nc")))
        if not files:
            raise FileNotFoundError(f"No files found for {var_name} in {var_dir}")
        years = {parse_year_from_filename(f) for f in files}
        year_sets.append(years)
    return sorted(set.intersection(*year_sets))

def expected_output_files(OUT_DIR, nyrs):
    return {
        "all_waves": os.path.join(OUT_DIR, "all_waves", f"EPFLUX_all_waves_{nyrs}yr_time_plev_lat.nc"),
        "wave1":     os.path.join(OUT_DIR, "wave1",     f"EPFLUX_wave1_{nyrs}yr_time_plev_lat.nc"),
        "wave2":     os.path.join(OUT_DIR, "wave2",     f"EPFLUX_wave2_{nyrs}yr_time_plev_lat.nc"),
        "wave_rest": os.path.join(OUT_DIR, "wave_rest", f"EPFLUX_wave_rest_{nyrs}yr_time_plev_lat.nc"),
    }

def process_one_year(args):
    """
    读取已经 timefixed 的年度文件，不再做人为时间平移。
    输出完整 EP flux 场：time × plev × lat
    注意：ComputeEPfluxDiv 本身就是 zonal-mean 量，因此没有 lon 维。
    """
    year_int, U_DIR, V_DIR, T_DIR, OMEGA_DIR = args

    try:
        fU = file_for(U_DIR, year_int, "U")
        fV = file_for(V_DIR, year_int, "V")
        fT = file_for(T_DIR, year_int, "T")
        fW = file_for(OMEGA_DIR, year_int, "OMEGA")

        time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

        with xr.open_dataset(fU, decode_times=time_coder) as dsU, \
             xr.open_dataset(fV, decode_times=time_coder) as dsV, \
             xr.open_dataset(fT, decode_times=time_coder) as dsT, \
             xr.open_dataset(fW, decode_times=time_coder) as dsW:

            if dsU.sizes.get("time", 0) < 2:
                return f"⚠️ Too few timesteps for year {year_int:04d}"

            # 1) hybrid -> pressure
            p_mid = compute_pressure_mid(dsU)

            # 2) log-pressure interpolation
            u_std = interp_profile_logp_4d(dsU["U"], p_mid, PLEV_STD_PA)
            v_std = interp_profile_logp_4d(dsV["V"], p_mid, PLEV_STD_PA)
            t_std = interp_profile_logp_4d(dsT["T"], p_mid, PLEV_STD_PA)
            w_std = interp_profile_logp_4d(dsW["OMEGA"] / 100.0, p_mid, PLEV_STD_PA)  # Pa/s -> hPa/s

            # 3) to numpy, dims: time, plev, lat, lon
            u_np = u_std.transpose("time", "plev", "lat", "lon").values
            v_np = v_std.transpose("time", "plev", "lat", "lon").values
            t_np = t_std.transpose("time", "plev", "lat", "lon").values
            w_np = w_std.transpose("time", "plev", "lat", "lon").values
            lat_np = dsU["lat"].values

            # 4) all waves
            ep1_all, ep2_all, div1_all, div2_all = ComputeEPfluxDiv(
                lat=lat_np,
                pres=(PLEV_STD_PA / 100.0),  # hPa
                u=u_np, v=v_np, t=t_np, w=w_np,
                do_ubar=DO_UBAR, wave=WAVE_ALL
            )

            # 5) wave-1
            ep1_w1, ep2_w1, div1_w1, div2_w1 = ComputeEPfluxDiv(
                lat=lat_np,
                pres=(PLEV_STD_PA / 100.0),  # hPa
                u=u_np, v=v_np, t=t_np, w=w_np,
                do_ubar=DO_UBAR, wave=WAVE_1
            )

            # 6) wave-2
            ep1_w2, ep2_w2, div1_w2, div2_w2 = ComputeEPfluxDiv(
                lat=lat_np,
                pres=(PLEV_STD_PA / 100.0),  # hPa
                u=u_np, v=v_np, t=t_np, w=w_np,
                do_ubar=DO_UBAR, wave=WAVE_2
            )

            # 7) remaining waves = all - wave1 - wave2
            # 这样避免使用 aostools_functions.py 里 wave=list 分支
            ep1_rest  = ep1_all  - ep1_w1  - ep1_w2
            ep2_rest  = ep2_all  - ep2_w1  - ep2_w2
            div1_rest = div1_all - div1_w1 - div1_w2
            div2_rest = div2_all - div2_w1 - div2_w2

            coords = {
                "time": dsU["time"],
                "plev": PLEV_STD_PA,
                "lat":  lat_np,
            }

            ds_out = xr.Dataset(
                data_vars={
                    "ep1_all":   (("time", "plev", "lat"), ep1_all),
                    "ep2_all":   (("time", "plev", "lat"), ep2_all),
                    "div1_all":  (("time", "plev", "lat"), div1_all),
                    "div2_all":  (("time", "plev", "lat"), div2_all),

                    "ep1_wave1":  (("time", "plev", "lat"), ep1_w1),
                    "ep2_wave1":  (("time", "plev", "lat"), ep2_w1),
                    "div1_wave1": (("time", "plev", "lat"), div1_w1),
                    "div2_wave1": (("time", "plev", "lat"), div2_w1),

                    "ep1_wave2":  (("time", "plev", "lat"), ep1_w2),
                    "ep2_wave2":  (("time", "plev", "lat"), ep2_w2),
                    "div1_wave2": (("time", "plev", "lat"), div1_w2),
                    "div2_wave2": (("time", "plev", "lat"), div2_w2),

                    "ep1_rest":   (("time", "plev", "lat"), ep1_rest),
                    "ep2_rest":   (("time", "plev", "lat"), ep2_rest),
                    "div1_rest":  (("time", "plev", "lat"), div1_rest),
                    "div2_rest":  (("time", "plev", "lat"), div2_rest),
                },
                coords=coords
            )

        del u_std, v_std, t_std, w_std, u_np, v_np, t_np, w_np
        gc.collect()

        return ds_out

    except Exception as e:
        return f"Error in year {year_int:04d}: {type(e).__name__}: {str(e)}"

def split_and_write_outputs(ds_full, out_files, root_dir, prefix_u):
    # all waves
    ds_all = ds_full[["ep1_all", "ep2_all", "div1_all", "div2_all"]].rename({
        "ep1_all": "ep1",
        "ep2_all": "ep2",
        "div1_all": "div1",
        "div2_all": "div2",
    })
    ds_all.attrs = {
        "description": "EP flux full field (time, plev, lat), all waves",
        "root_dir": root_dir,
        "file_prefix": prefix_u,
        "units_ep1": "m2/s2",
        "units_ep2": "hPa*m/s2",
        "units_div1": "m/s/day",
        "units_div2": "m/s/day",
        "history": "Computed from *_timefixed files; no additional internal time shift; all zonal waves included"
    }

    # wave1
    ds_w1 = ds_full[["ep1_wave1", "ep2_wave1", "div1_wave1", "div2_wave1"]].rename({
        "ep1_wave1": "ep1",
        "ep2_wave1": "ep2",
        "div1_wave1": "div1",
        "div2_wave1": "div2",
    })
    ds_w1.attrs = {
        "description": "EP flux full field (time, plev, lat), wave-1 only",
        "root_dir": root_dir,
        "file_prefix": prefix_u,
        "units_ep1": "m2/s2",
        "units_ep2": "hPa*m/s2",
        "units_div1": "m/s/day",
        "units_div2": "m/s/day",
        "history": "Computed from *_timefixed files; no additional internal time shift; wave=1"
    }

    # wave2
    ds_w2 = ds_full[["ep1_wave2", "ep2_wave2", "div1_wave2", "div2_wave2"]].rename({
        "ep1_wave2": "ep1",
        "ep2_wave2": "ep2",
        "div1_wave2": "div1",
        "div2_wave2": "div2",
    })
    ds_w2.attrs = {
        "description": "EP flux full field (time, plev, lat), wave-2 only",
        "root_dir": root_dir,
        "file_prefix": prefix_u,
        "units_ep1": "m2/s2",
        "units_ep2": "hPa*m/s2",
        "units_div1": "m/s/day",
        "units_div2": "m/s/day",
        "history": "Computed from *_timefixed files; no additional internal time shift; wave=2"
    }

    # rest
    ds_rest = ds_full[["ep1_rest", "ep2_rest", "div1_rest", "div2_rest"]].rename({
        "ep1_rest": "ep1",
        "ep2_rest": "ep2",
        "div1_rest": "div1",
        "div2_rest": "div2",
    })
    ds_rest.attrs = {
        "description": "EP flux full field (time, plev, lat), remaining waves = all - wave1 - wave2",
        "root_dir": root_dir,
        "file_prefix": prefix_u,
        "units_ep1": "m2/s2",
        "units_ep2": "hPa*m/s2",
        "units_div1": "m/s/day",
        "units_div2": "m/s/day",
        "history": "Computed from *_timefixed files; no additional internal time shift; rest = all - wave1 - wave2"
    }

    os.makedirs(os.path.dirname(out_files["all_waves"]), exist_ok=True)
    os.makedirs(os.path.dirname(out_files["wave1"]), exist_ok=True)
    os.makedirs(os.path.dirname(out_files["wave2"]), exist_ok=True)
    os.makedirs(os.path.dirname(out_files["wave_rest"]), exist_ok=True)

    print(f"💾 Writing: {out_files['all_waves']}")
    ds_all.to_netcdf(out_files["all_waves"])

    print(f"💾 Writing: {out_files['wave1']}")
    ds_w1.to_netcdf(out_files["wave1"])

    print(f"💾 Writing: {out_files['wave2']}")
    ds_w2.to_netcdf(out_files["wave2"])

    print(f"💾 Writing: {out_files['wave_rest']}")
    ds_rest.to_netcdf(out_files["wave_rest"])

def run_one_root(ROOT_DIR):
    U_DIR     = os.path.join(ROOT_DIR, "U")
    V_DIR     = os.path.join(ROOT_DIR, "V")
    T_DIR     = os.path.join(ROOT_DIR, "T")
    OMEGA_DIR = os.path.join(ROOT_DIR, "OMEGA")
    OUT_DIR   = os.path.join(ROOT_DIR, "EPflux_daily")
    os.makedirs(OUT_DIR, exist_ok=True)

    years = get_available_years(U_DIR, V_DIR, T_DIR, OMEGA_DIR)
    if not years:
        print(f"❌ No common years found under {ROOT_DIR}")
        return

    nyrs = len(years)
    out_files = expected_output_files(OUT_DIR, nyrs)

    existing = [fp for fp in out_files.values() if os.path.exists(fp) and os.path.getsize(fp) > 0]
    if existing and not OVERWRITE:
        print(f"\n⏭️ Detected existing EPflux output, skip run:")
        print(f"ROOT_DIR = {ROOT_DIR}")
        for fp in existing:
            print("   ", fp)
        return

    prefix_u = detect_prefix(U_DIR, "U")

    print(f"\n🚀 EPflux full-field from timefixed files")
    print(f"ROOT_DIR   = {ROOT_DIR}")
    print(f"Prefix     = {prefix_u}")
    print(f"Years      = {years[0]}–{years[-1]}  (n={nyrs})")
    print(f"Workers    = {MAX_WORKERS}")
    print(f"Outputs    =")
    for k, v in out_files.items():
        print(f"   {k}: {v}")

    args_list = [(y, U_DIR, V_DIR, T_DIR, OMEGA_DIR) for y in years]
    pool_results = []

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        fut = {ex.submit(process_one_year, args): args[0] for args in args_list}
        for f in tqdm(as_completed(fut), total=len(fut), desc=f"EPflux {os.path.basename(ROOT_DIR)}"):
            res = f.result()
            if isinstance(res, xr.Dataset):
                pool_results.append(res)
            elif res is not None:
                print(f"⚠️ {res}")

    if not pool_results:
        print("❌ No data collected.")
        return

    print("📊 Finalizing: concat + sortby(time) + drop duplicate time ...")
    ds_full = xr.concat(pool_results, dim="time").sortby("time")

    tvals = ds_full["time"].values
    _, idx = np.unique(tvals, return_index=True)
    idx = np.sort(idx)
    ds_full = ds_full.isel(time=idx)

    split_and_write_outputs(ds_full, out_files, ROOT_DIR, prefix_u)
    print("✅ Done!")

def main():
    for root in ROOT_DIRS:
        run_one_root(root)

if __name__ == "__main__":
    main()


⏭️ Detected existing EPflux output, skip run:
ROOT_DIR = /mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed
    /mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed/EPflux_daily/all_waves/EPFLUX_all_waves_210yr_time_plev_lat.nc
    /mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed/EPflux_daily/wave1/EPFLUX_wave1_210yr_time_plev_lat.nc
    /mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed/EPflux_daily/wave2/EPFLUX_wave2_210yr_time_plev_lat.nc
    /mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed/EPflux_daily/wave_rest/EPFLUX_wave_rest_210yr_time_plev_lat.nc

⏭️ Detected existing EPflux output, skip run:
ROOT_DIR = /mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002_timefixed
    /mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002_timefixed/EPflux_daily/all_waves/EPFLUX_all_waves_205yr_time_plev_lat.nc
    /mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002_timefixed/EPflux_daily/wave1/EPFLUX_wave1_205yr_time_plev_lat.nc
    /mnt/soclim0/public_data/weiji

EPflux BWCN:   4%|▍         | 1/24 [00:02<01:05,  2.83s/it]

⚠️ ⚠️ Too few timesteps for year 0024


/home/weiji/restart_exam/code_cleaned/Longrun/date_treatment/aostools_functions.py:86: RuntimeWarning: Mean of empty slice
  v_bar = np.nanmean(v,axis=-1)
/home/weiji/restart_exam/code_cleaned/Longrun/date_treatment/aostools_functions.py:87: RuntimeWarning: Mean of empty slice
  t_bar = np.nanmean(t,axis=-1) # t_bar = theta_bar
/home/weiji/restart_exam/code_cleaned/Longrun/date_treatment/aostools_functions.py:92: RuntimeWarning: Mean of empty slice
  dthdp = np.nanmean(dthdp,axis=0)[np.newaxis,:]
/home/weiji/restart_exam/code_cleaned/Longrun/date_treatment/aostools_functions.py:99: RuntimeWarning: Mean of empty slice
  t = np.nanmean(v*t,axis=-1) # t = bar(v'Th')
/home/weiji/restart_exam/code_cleaned/Longrun/date_treatment/aostools_functions.py:167: RuntimeWarning: Mean of empty slice
  upvp = np.nanmean(u*v,axis=-1)
/home/weiji/restart_exam/code_cleaned/Longrun/date_treatment/aostools_functions.py:187: RuntimeWarning: Mean of empty slice
  w = np.nanmean(w*u,axis=-1) # w = bar(u'w') [

📊 Finalizing: concat + sortby(time) + drop duplicate time ...
💾 Writing: /mnt/soclim0/public_data/weiji/BWCN/EPflux_daily/all_waves/EPFLUX_all_waves_24yr_time_plev_lat.nc
💾 Writing: /mnt/soclim0/public_data/weiji/BWCN/EPflux_daily/wave1/EPFLUX_wave1_24yr_time_plev_lat.nc
💾 Writing: /mnt/soclim0/public_data/weiji/BWCN/EPflux_daily/wave2/EPFLUX_wave2_24yr_time_plev_lat.nc
💾 Writing: /mnt/soclim0/public_data/weiji/BWCN/EPflux_daily/wave_rest/EPFLUX_wave_rest_24yr_time_plev_lat.nc
✅ Done!
